In [ ]:
import os
import dotenv
import mysql.connector as mc
import pandas as pd

dotenv.load_dotenv()

DB_CONFIG = {
    'host': os.getenv('DB_HOST', '127.0.0.1'),
    'port': int(os.getenv('DB_PORT', 3306)),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_DATABASE'),
    'autocommit': False,
}
conn = mc.connect(**DB_CONFIG)
query = """
SELECT
    t.roadName,
    SUM(
        t.hour00 + t.hour01 + t.hour02 + t.hour03 +
        t.hour04 + t.hour05 + t.hour06 + t.hour07 +
        t.hour08 + t.hour09 + t.hour10 + t.hour11 +
        t.hour12 + t.hour13 + t.hour14 + t.hour15 +
        t.hour16 + t.hour17 + t.hour18 + t.hour19 +
        t.hour20 + t.hour21 + t.hour22 + t.hour23
    ) AS total_traffic,
    AVG(h.hazard_grade) AS avg_hazard
FROM traffic_raw t
JOIN hazard_raw h
ON t.roadName = h.road_name
GROUP BY t.roadName
"""

df = pd.read_sql(query, conn)

In [ ]:
import os
import dotenv
import mysql.connector as mc
import pandas as pd

dotenv.load_dotenv()

DB_CONFIG = {
    'host': os.getenv('DB_HOST', '127.0.0.1'),
    'port': int(os.getenv('DB_PORT', 3306)),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_DATABASE'),
    'autocommit': False,
}
conn = mc.connect(**DB_CONFIG)

query = """
SELECT
    t.roadName,
    t.linkID,
    SUM(
        t.hour00 + t.hour01 + t.hour02 + t.hour03 +
        t.hour04 + t.hour05 + t.hour06 + t.hour07 +
        t.hour08 + t.hour09 + t.hour10 + t.hour11 +
        t.hour12 + t.hour13 + t.hour14 + t.hour15 +
        t.hour16 + t.hour17 + t.hour18 + t.hour19 +
        t.hour20 + t.hour21 + t.hour22 + t.hour23
    ) AS total_traffic,
    AVG(h.hazard_grade) AS avg_hazard
FROM traffic_raw t
JOIN hazard_raw h
ON t.linkID = h.link_id
GROUP BY t.linkID, t.roadName
"""

df = pd.read_sql(query, conn)

In [ ]:
import matplotlib.pyplot as plt

df['stress_score'] = df['total_traffic'] * 0.7 + df['avg_hazard'] * 0.3

top10 = df.sort_values(by='stress_score', ascending=False).head(10)

plt.figure()
plt.barh(top10['roadName'], top10['stress_score'])
plt.xlabel("Stress Score")
plt.title("Top 10 Stress Roads")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure()
plt.scatter(df['total_traffic'], df['avg_hazard'])
plt.xlabel("Traffic Volume")
plt.ylabel("Hazard Level")
plt.title("Traffic vs Hazard")
plt.show()